In [0]:
%run "./00_config"

## 01. Core Functions

- `get_pending_files()` - reads `Is_Active = 1` rows from the control table (optionally filtered by
  `DataMart`), builds the full source path, and excludes any file already logged as `success`
  in the log table for the same target (unless `force_reload = True`).
- `write_log()` - appends a single log row immediately (failure won't break the process).
  Uses an explicit schema (`log_schema`) rather than type inference
- `base_log_record()` - maps control-table columns to the log-table's internal fields.


In [0]:
def get_pending_files():
    control_df = spark.table(control_table).filter(col("Is_Active") == 1)

    if datamart_filter is not None:
        control_df = control_df.filter(col("DataMart") == datamart_filter)

    if force_reload:
        return control_df.collect()

    succeeded_df = (spark.table(log_table)
                    .filter(col("status") == "SUCCESS")
                    .select("file_name", "tgt_catalog", "tgt_schema", "tgt_table")
                    .distinct())

    pending_df = (control_df.join(
                      succeeded_df,
                      (control_df.Source_File_Name == succeeded_df.file_name) &
                      (control_df.Target_Catalog == succeeded_df.tgt_catalog) &
                      (control_df.Target_Schema == succeeded_df.tgt_schema) &
                      (control_df.Target_Table_Name == succeeded_df.tgt_table),
                      "left_anti"))

    return pending_df.collect()


In [0]:
def write_log(record: dict):

    ordered = {field.name: record.get(field.name) for field in log_schema.fields}
    log_df = spark.createDataFrame([Row(**ordered)], schema=log_schema)
    log_df.write.format("delta").mode("append").saveAsTable(log_table)

In [0]:
def base_log_record(r, batch_id, full_source_path):
    return {
        "log_id": str(uuid.uuid4()),
        "batch_id": batch_id,
        "datamart": r["DataMart"],
        "file_name": r["Source_File_Name"],
        "source_path": full_source_path,
        "tgt_catalog": r["Target_Catalog"],
        "tgt_schema": r["Target_Schema"],
        "tgt_table": r["Target_Table_Name"],
        "expected_count": r["File_Count"],
    }